In [19]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/priyadharsana/phishunter-url-dataset/phishing_url_dataset.csv


In [20]:
url_df = pd.read_csv("/kaggle/input/datasets/priyadharsana/phishunter-url-dataset/phishing_url_dataset.csv")
print(url_df.head())
print(url_df['label'].value_counts())

                                                 url  label        source
0  https://listman.redhat.com/mailman/listinfo/ex...      0  Assassin.csv
1  http://us.click.yahoo.com/pt6ybb/nxieaa/mg3haa...      0  Assassin.csv
2                  http://docs.yahoo.com/info/terms/      0  Assassin.csv
3  http://www.pcworld.com/news/article/0aid103259...      0  Assassin.csv
4           http://tb.tf/mailman/listinfo/irregulars      0  Assassin.csv
label
0    52603
1    20615
Name: count, dtype: int64


In [21]:
def extract_url_features(url):
    url = str(url)
    
    features = {}
    
    # Length
    features['url_length'] = len(url)
    
    # Digits
    features['num_digits'] = sum(c.isdigit() for c in url)
    
    # Special chars
    features['num_special'] = sum(not c.isalnum() for c in url)
    
    # HTTPS
    features['has_https'] = int('https' in url)
    
    # Subdomains
    features['num_dots'] = url.count('.')
    
    # Hyphens
    features['num_hyphens'] = url.count('-')
    
    # 🔥 NEW FEATURES
    
    # Suspicious keywords
    suspicious_words = ['login', 'verify', 'bank', 'secure', 'account', 'update', 'free', 'bonus']
    features['has_suspicious_word'] = int(any(word in url for word in suspicious_words))
    
    # IP address
    features['has_ip'] = int(bool(re.search(r'\d+\.\d+\.\d+\.\d+', url)))
    
    # URL length > 75
    features['is_long_url'] = int(len(url) > 75)
    
    # Contains '@'
    features['has_at_symbol'] = int('@' in url)
    
    # Count '='
    features['num_params'] = url.count('=')
    
    # Count '/'
    features['num_slashes'] = url.count('/')
    
    return features

In [22]:
feature_list = []

for url in url_df['url']:
    feature_list.append(extract_url_features(url))

X = pd.DataFrame(feature_list)
y = url_df['label']

print(X.head())

   url_length  num_digits  num_special  has_https  num_dots  num_hyphens  \
0          56           0            9          1         2            1   
1          56           3           11          0         3            0   
2          33           0            8          0         2            0   
3          52           9            9          0         3            0   
4          40           0            7          0         1            0   

   has_suspicious_word  has_ip  is_long_url  has_at_symbol  num_params  \
0                    0       0            0              0           0   
1                    0       0            0              0           0   
2                    0       0            0              0           0   
3                    0       0            0              0           0   
4                    0       0            0              0           0   

   num_slashes  
0            5  
1            7  
2            5  
3            5  
4            

In [23]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [28]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score

rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=10,
    class_weight={0:1, 1:2},
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

y_pred = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.856733133023764
              precision    recall  f1-score   support

           0       0.93      0.87      0.90     10521
           1       0.71      0.83      0.76      4123

    accuracy                           0.86     14644
   macro avg       0.82      0.85      0.83     14644
weighted avg       0.87      0.86      0.86     14644



In [29]:
import joblib
joblib.dump(rf_model,"url_rf_model.pkl")

['url_rf_model.pkl']